### 1. Conectar con Google Drive e Importar Librerías
Ejecuta la siguiente celda para dar acceso a Colab a tus archivos de Drive e importar `pandas` y `numpy`.

In [10]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Configurar la Ruta y Listar Archivos
Por defecto, al montar Google Drive, tu unidad principal se encuentra en `/content/drive/My Drive/`.

IMPORTANTE => Hacer un acceso directo de la carpeta Datos a tu unidad para que el programa pueda leer los archivos.

Estructura de carpetas dentro de `Datos`:
- `Datos/Crudo`: datos originales (no se usan en este notebook).
- `Datos/Procesado`: datos limpios y consolidados que exportó el notebook de Data Understanding (sección 25). **Esta es la carpeta de entrada de este notebook.**

In [11]:
RUTA_DATOS = '/content/drive/My Drive/Datos/Procesado'

if os.path.isdir(RUTA_DATOS):
    print(f"¡Éxito! Carpeta detectada en: {RUTA_DATOS}\n")
    archivos = sorted(f for f in os.listdir(RUTA_DATOS) if f.endswith(('.parquet', '.csv')))
    print(f"Se encontraron {len(archivos)} archivos de datos procesados:")
    for i, archivo in enumerate(archivos, 1):
        peso = os.path.getsize(os.path.join(RUTA_DATOS, archivo)) / 1024**2
        print(f"{i}. {archivo} ({peso:.1f} MB)")
else:
    print("Aún no se detectó la carpeta. Por favor verifica que exista 'Mi Unidad/Datos/Procesado' "
          "(se crea al ejecutar la sección 25 del notebook de Data Understanding) y vuelve a ejecutar esta celda.")

¡Éxito! Carpeta detectada en: /content/drive/My Drive/Datos/Procesado

Se encontraron 8 archivos de datos procesados:
1. concentrado_consumido.parquet (0.7 MB)
2. consolidado_marzo_junio_2025.parquet (0.2 MB)
3. produccion_leche_sesion.parquet (13.0 MB)
4. s0_base_limpia.parquet (19.3 MB)
5. s1_ventanas_robot.parquet (1.6 MB)
6. s2_sesiones_etapa.parquet (5.0 MB)
7. s3_episodios_cola.parquet (2.0 MB)
8. s4_vaca_dia.parquet (0.9 MB)


### 3. Lectura de los datos procesados
Se leen los archivos exportados en la sección 25 de Data Understanding. Cada archivo se carga en el DataFrame que usa el resto del notebook:

| Archivo | DataFrame | Contenido |
|---|---|---|
| `consolidado_marzo_junio_2025` | `df_consolidado` | Archivos de marzo-junio 2025 unidos y ordenados por `hora_inicio` |
| `concentrado_consumido` | `concentradoConsumido` | Consumo de concentrado por vaca |
| `produccion_leche_sesion` | `prodLecheSesion` | Producción por sesión (histórico) con tipos convertidos, sin filtrar |
| `s0_base_limpia` | `s0` | Ordeños filtrados + variables temporales y de lactancia |
| `s1_ventanas_robot` | `s1` | Robot × ventana de 30 min |
| `s2_sesiones_etapa` | `s2` | Ordeños con días en leche conocidos |
| `s3_episodios_cola` | `s3` | Episodios de cola |
| `s4_vaca_dia` | `s4` | Vaca × día |

Funciona con archivos `.parquet` o `.csv` (según el `FORMATO_EXPORTACION` que se haya usado al exportar).

In [12]:
ARCHIVOS = {
    # nombre del archivo (sin extensión): nombre del DataFrame
    "consolidado_marzo_junio_2025": "df_consolidado",
    "concentrado_consumido": "concentradoConsumido",
    "produccion_leche_sesion": "prodLecheSesion",
    "s0_base_limpia": "s0",
    "s1_ventanas_robot": "s1",
    "s2_sesiones_etapa": "s2",
    "s3_episodios_cola": "s3",
    "s4_vaca_dia": "s4",
}

def leer_procesado(nombre):
    """Lee un archivo de Datos/Procesado (parquet o csv). Devuelve None si no existe."""
    for ext in ("parquet", "csv"):
        ruta = os.path.join(RUTA_DATOS, f"{nombre}.{ext}")
        if os.path.exists(ruta):
            return pd.read_parquet(ruta) if ext == "parquet" else pd.read_csv(ruta, low_memory=False)
    return None

print("Iniciando la lectura de archivos...")
datos = {}
for nombre, variable in ARCHIVOS.items():
    d = leer_procesado(nombre)
    if d is None:
        print(f"No se encontró '{nombre}' en {RUTA_DATOS}")
        continue
    datos[variable] = d
    print(f"✔ {variable:<22} <- {nombre}: {len(d):,} filas x {d.shape[1]} columnas")

print(f"\nArchivos cargados correctamente: {len(datos)} de {len(ARCHIVOS)}")

Iniciando la lectura de archivos...
✔ df_consolidado         <- consolidado_marzo_junio_2025: 7,249 filas x 37 columnas
✔ concentradoConsumido   <- concentrado_consumido: 130,112 filas x 6 columnas
✔ prodLecheSesion        <- produccion_leche_sesion: 278,767 filas x 50 columnas
✔ s0                     <- s0_base_limpia: 278,167 filas x 70 columnas
✔ s1                     <- s1_ventanas_robot: 180,837 filas x 19 columnas
✔ s2                     <- s2_sesiones_etapa: 262,618 filas x 19 columnas
✔ s3                     <- s3_episodios_cola: 57,939 filas x 19 columnas
✔ s4                     <- s4_vaca_dia: 114,910 filas x 14 columnas

Archivos cargados correctamente: 8 de 8


### 4. Restaurar tipos de datos
Al guardar en Drive algunos tipos cambian (`mes` se exporta como texto `YYYY-MM`, y las columnas con tipos mezclados se guardan como texto; con CSV todas las fechas vuelven como texto). Aquí se restauran:
- `mes` -> periodo mensual.
- `fecha` -> fecha (`datetime64`, a las 00:00 h).
- `hora_inicio`, `fin`, `inicio` y `ventana` -> fecha y hora.

In [13]:
COLS_FECHA_HORA = ["hora_inicio", "fin", "inicio", "ventana"]

def restaurar_tipos(d):
    d = d.copy()
    if "mes" in d.columns:
        d["mes"] = pd.to_datetime(d["mes"], errors="coerce").dt.to_period("M")
    if "fecha" in d.columns:
        d["fecha"] = pd.to_datetime(d["fecha"], errors="coerce")
    for c in COLS_FECHA_HORA:
        if c in d.columns:
            d[c] = pd.to_datetime(d[c], errors="coerce")
    return d

datos = {variable: restaurar_tipos(d) for variable, d in datos.items()}

# Dejar cada DataFrame como variable del notebook
df_consolidado = datos.get("df_consolidado")
concentradoConsumido = datos.get("concentradoConsumido")
prodLecheSesion = datos.get("prodLecheSesion")
s0, s1, s2, s3, s4 = (datos.get(k) for k in ["s0", "s1", "s2", "s3", "s4"])

print("Tipos restaurados.")

Tipos restaurados.
